# Rally 12B Scorecard

Score direct Gemma4 12B Heretic against the A100/B75 RP candidate on Kaggle. Includes a 100-prompt adult false-refusal probe (target: direct ~100/100, RP ~6/100). Serving path is vLLM, not WebGPU export.

In [ ]:
import os, platform, shutil
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('input_dirs=', [str(p) for p in Path('/kaggle/input').glob('*')])
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        print(f'gpu_{i}=', props.name, round(props.total_memory / 1024**3, 2), 'GiB', f'sm_{major}{minor}')
except Exception as exc:
    print('torch_probe_error=', repr(exc))

In [ ]:
import os, subprocess, sys, time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['RALLY_SCORECARD_LOAD_IN_4BIT'] = '1'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

packages = [
    'git+https://github.com/huggingface/transformers.git',
    'accelerate>=1.13.0',
    'peft>=0.19.0',
    'safetensors>=0.7.0',
    'huggingface_hub[hf_transfer]>=1.5.0',
    'bitsandbytes>=0.49.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())
import shutil, sys
subprocess.check_call([sys.executable, str(REPO_DIR / 'scripts/kaggle_disk_cleanup.py'), '--root', '/kaggle/working'])
shutil.rmtree(REPO_DIR / '.git', ignore_errors=True)
print('working_disk_free_gb_after_cleanup=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path(os.environ.get('RALLY_SCORECARD_WORK_DIR', '/kaggle/working/rally-12b-scorecard'))
REPORT_PATH = WORK_DIR / 'rally-12b-scorecard-report.json'
sweep_candidates = os.environ.get('RALLY_SCORECARD_SWEEP', '').strip()
cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/kaggle_rally_e2b_scorecard.py'),
    '--work-dir', str(WORK_DIR),
    '--report-path', str(REPORT_PATH),
    '--artifact-name', os.environ.get('RALLY_TWO_STAGE_ARTIFACT_NAME', 'rally-12b-two-stage-sft'),
    '--direct-model-id', os.environ.get('RALLY_DIRECT_MODEL_ID', 'igorls/gemma-4-12B-it-heretic'),
    '--candidate-name', os.environ.get('RALLY_CANDIDATE_NAME', 'a100-b75'),
    '--stage-b-scale', os.environ.get('RALLY_STAGE_B_SCALE', '0.75'),
    '--max-new-tokens', os.environ.get('RALLY_SCORECARD_MAX_TOKENS', '32'),
    '--temperature', os.environ.get('RALLY_SCORECARD_TEMPERATURE', '0.2'),
    '--min-total', os.environ.get('RALLY_SCORECARD_MIN_TOTAL', '0.70'),
    '--min-margin', os.environ.get('RALLY_SCORECARD_MIN_MARGIN', '0.05'),
    '--refusal-probe-count', os.environ.get('RALLY_REFUSAL_PROBE_COUNT', '100'),
    '--max-false-refusal-rate', os.environ.get('RALLY_MAX_FALSE_REFUSAL_RATE', '0.10'),
]
if sweep_candidates:
    cmd.extend(['--sweep-candidates', sweep_candidates])
if os.environ.get('RALLY_KEEP_MERGED', '0') == '1':
    cmd.append('--keep-merged')
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import json, os

report_path = Path(os.environ.get('RALLY_SCORECARD_WORK_DIR', '/kaggle/working/rally-12b-scorecard')) / 'rally-12b-scorecard-report.json'
print('report_path=', report_path)
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({
        'ok': report.get('ok'),
        'artifact_dir': report.get('artifact_dir'),
        'direct_total': report.get('scores', {}).get('direct', {}).get('total'),
        'rp_total': report.get('scores', {}).get('rp', {}).get('total'),
        'best_candidate': report.get('best_candidate'),
        'promotion_decision': report.get('promotion_decision'),
        'refusal_probe_direct': (report.get('refusal_probe') or {}).get('direct'),
        'refusal_probe_rp': report.get('refusal_probe', {}).get(os.environ.get('RALLY_CANDIDATE_NAME', 'a100-b75')),
    }, indent=2))